In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Fetch Data via ChEMBL API

- Using official Python client `chembl_webresource_client` for accessing ChEMBL API
- Fetch data for all molecules binding to a certain target defined by its UniProt ID, e.g.:
    - 3-hydroxy-3-methylglutaryl-coenzyme A reductase (HMGCR): P04035
    - Cystic fibrosis transmembrane conductance regulator (CFTR): P13569
    - Epidermal growth factor receptor (EGFR): P00533 

In [ ]:
from chembl_webresource_client.new_client import new_client

# Choose target based on UniProt ID
uniprot_id = "P04035"

# API access for target, bioactivity data (pIC50 values), and the identified binding drugs/ligands
targets_api = new_client.target
bioactivities_api = new_client.activity
compounds_api = new_client.molecule

In [ ]:
# Fetch all targets from ChEMBL database that are associated with the above UniProt ID
targets = targets_api.get(target_components__accession=uniprot_id).only(
    "target_chembl_id", 
    "pref_name", 
    "organism", 
    "target_type"
    )

targets_df = pd.DataFrame.from_records(targets)

print(f"Found {len(targets_df)} targets for UniProt ID {uniprot_id}:")
targets_df

In [ ]:
# Select the target within ChEMBL database
chembl_id = targets_df.iloc[0]["target_chembl_id"]
chembl_id

In [ ]:
bioactivities = bioactivities_api.filter(
    target_chembl_id=chembl_id, assay_type="B", type="IC50", relation="=", target_organism="Homo sapiens"
    ).only(
        "activity_id",
        "assay_chembl_id",
        "assay_description",
        "molecule_chembl_id",
        "type",
        "standard_value",
        "standard_units",
    )
        
bioactivities_df = pd.DataFrame.from_records(bioactivities)

bioactivities_df = bioactivities_df.dropna()
bioactivities_df = bioactivities_df[bioactivities_df["standard_units"] == "nM"]
bioactivities_df = bioactivities_df.drop_duplicates("molecule_chembl_id")
bioactivities_df = bioactivities_df.reset_index().drop(columns=["index"])
bioactivities_df['standard_value'] = bioactivities_df['standard_value'].astype(float)
bioactivities_df['pIC50'] = -np.log10( bioactivities_df['standard_value']*10**(-9) )
bioactivities_df = bioactivities_df.drop(columns=["units", "value", "standard_units"])

print(f"Found {len(bioactivities_df)} different ligands with available bioactivity for target with ID {chembl_id}:")
bioactivities_df

In [ ]:
compounds = compounds_api.filter(molecule_chembl_id__in=list(bioactivities_df["molecule_chembl_id"])).only(
    "molecule_chembl_id",
    "molecule_structures",
)

compounds_df = pd.DataFrame.from_records(compounds)

compounds_df = compounds_df.dropna()
compounds_df = compounds_df.drop_duplicates("molecule_chembl_id")
compounds_df["canonical_smiles"] = compounds_df["molecule_structures"].apply(lambda x: list(x.values())[0])
compounds_df = compounds_df.reset_index().drop(columns=["index"])

print(f"Created SMILES for {len(compounds_df)} different ligands for target with ID {chembl_id}:")
compounds_df

In [ ]:
merged_df = pd.merge(bioactivities_df, compounds_df, 
                     on="molecule_chembl_id").drop(columns=
                                                   ["molecule_structures", 
                                                    "activity_id", 
                                                    "assay_description",
                                                    "type"])
merged_df

In [ ]:
# Save compounds (described via both molecule_chembl_id and canonical_smiles) with associated pIC50 values towards selected target protein to a csv file
merged_df.to_csv(f"../data/{chembl_id}.csv")

In [ ]:
# Have a first glance at the pIC50 value distribution of the identified target-binding cpds
plt.figure(figsize=(9,6))
# pIC50 below 6 indicates weak binding affinity
plt.hist([value for value in merged_df["pIC50"] if value <= 6], color="#C57E7E")
# pIC50 above 6 indicates strong binding affinity (i.e., drug-like behavior)
plt.hist([value for value in merged_df["pIC50"] if value > 6], color="#4CDB3FE1")

plt.title(f"pIC50 Value Distribution Towards Target {uniprot_id}")
plt.xlabel("pIC50 / $-$")
plt.ylabel("Count / $-$")

plt.show()